In [ ]:
# 1️ Install Required Libraries (Run once)
# !pip install pandas numpy librosa soundfile matplotlib seaborn tqdm openpyxl

In [2]:
# ============================================================
# 🧠 Task 1: Multi-Class Dysarthria Severity Classification
# Data Loading + Preprocessing Notebook - Train Set
# ============================================================
import os
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# --- 1. Define Paths ---
BASE_DIR = r"Train/task1"
META_PATH = os.path.join(BASE_DIR, "sand_task_1.xlsx")
TRAIN_DIR = os.path.join(BASE_DIR, "training")
os.makedirs(TRAIN_DIR, exist_ok=True) # Ensure existence

# --- 2. Load Metadata ---
metadata = pd.read_excel(META_PATH)
# Convert ID column to set for fast lookup later
valid_ids = set(metadata["ID"].astype(str).str.strip().values) 
print("✅ Metadata Loaded.")

# --- 3. Create DataFrame Linking Audio Paths with Metadata ---
audio_data = []
folders = sorted(os.listdir(TRAIN_DIR))

# Process all audio files and link with metadata
for folder in tqdm(folders, desc="Linking Train Audio"):
    folder_path = os.path.join(TRAIN_DIR, folder)
    if not os.path.isdir(folder_path): continue
    for file in os.listdir(folder_path):
        if file.lower().endswith(".wav"):
            file_id = file.split("_")[0] # e.g., 'ID000'
            if file_id in valid_ids:
                row = metadata[metadata["ID"] == file_id].iloc[0]
                audio_data.append({
                    "ID": file_id,
                    "Age": row["Age"],
                    "Sex": row["Sex"],
                    "Class": row["Class"],
                    "Task": folder,
                    "Filepath": os.path.join(folder_path, file)
                })

audio_df = pd.DataFrame(audio_data)
# Save combined dataset (initial)
audio_df.to_csv(os.path.join(BASE_DIR, "audio_metadata_train_raw.csv"), index=False)
print(f"\n✅ Train DataFrame Created (Rows: {len(audio_df)}). Raw CSV saved.")
print(audio_df.head())

✅ Metadata Loaded.


Linking Train Audio: 100%|██████████| 8/8 [00:00<00:00, 21.05it/s]


✅ Train DataFrame Created (Rows: 2176). Raw CSV saved.
      ID  Age Sex  Class        Task  \
0  ID000   80   M      5  phonationA   
1  ID001   61   F      5  phonationA   
2  ID002   51   F      4  phonationA   
3  ID003   59   M      3  phonationA   
4  ID005   80   F      5  phonationA   

                                            Filepath  
0  Train/task1\training\phonationA\ID000_phonatio...  
1  Train/task1\training\phonationA\ID001_phonatio...  
2  Train/task1\training\phonationA\ID002_phonatio...  
3  Train/task1\training\phonationA\ID003_phonatio...  
4  Train/task1\training\phonationA\ID005_phonatio...  


In [3]:
# --- 4. Exploratory Data Analysis (EDA) & Duration Calculation ---
print("--- Basic Dataset Info ---")
print(f"Total rows: {audio_df.shape[0]}")
print("Class Distribution:\n", audio_df['Class'].value_counts().sort_index())

# Calculate Duration (re-using efficient soundfile/librosa logic)
if 'Duration_s' not in audio_df.columns:
    durations = []
    # Use RelPath if available from a previous cell's output, otherwise use Filepath
    path_col = 'Filepath' 
    for p in tqdm(audio_df[path_col].values, desc="Computing Train Durations"):
        try:
            info = sf.info(p)
            durations.append(float(info.duration))
        except Exception:
            # Fallback to librosa (slower)
            try:
                y, sr_ = librosa.load(p, sr=None)
                durations.append(float(len(y) / sr_))
            except Exception:
                durations.append(np.nan) # Mark as missing
    audio_df['Duration_s'] = durations
    print("\n✅ Audio Durations Computed.")

# Summary statistics
print("\nAge Summary (years):\n", audio_df['Age'].describe())
print("\nDuration Summary (seconds):\n", audio_df['Duration_s'].describe())

# Calculate Spearman Correlation
audio_df['Class_num'] = audio_df['Class'].astype(int) # Ensure numeric class for correlation
corr_age_class_dur = audio_df[['Age','Class_num','Duration_s']].corr(method='spearman')
print("\nSpearman Correlation (Age, Class, Duration):\n", corr_age_class_dur)

# Per-class aggregations
agg = audio_df.groupby('Class_num').agg(
    samples=('ID','count'),
    mean_age=('Age','mean'),
    mean_duration=('Duration_s','mean')
).sort_index()

# Save EDA summary
agg.to_csv(os.path.join(BASE_DIR, "per_class_summary_train_raw.csv"), index=True)
print("\n💾 Per-class summary saved.")

--- Basic Dataset Info ---
Total rows: 2176
Class Distribution:
 1     48
2    208
3    456
4    608
5    856
Name: Class, dtype: int64


Computing Train Durations: 100%|██████████| 2176/2176 [00:00<00:00, 2693.98it/s]


✅ Audio Durations Computed.

Age Summary (years):
 count    2176.000000
mean       63.514706
std        11.258949
min        23.000000
25%        56.750000
50%        64.500000
75%        72.000000
max        89.000000
Name: Age, dtype: float64

Duration Summary (seconds):
 count    2176.000000
mean       13.818564
std         7.355231
min         0.720000
25%         8.480000
50%        12.400000
75%        17.930000
max        48.240000
Name: Duration_s, dtype: float64

Spearman Correlation (Age, Class, Duration):
                  Age  Class_num  Duration_s
Age         1.000000  -0.020064   -0.226279
Class_num  -0.020064   1.000000    0.298478
Duration_s -0.226279   0.298478    1.000000

💾 Per-class summary saved.


In [ ]:
# --- 5. Corrected Cleaning & Standardization (8kHz Mono PCM_16) ---
import hashlib # <--- ADDED MISSING IMPORT

CLEAN_DIR = os.path.join(BASE_DIR, "cleaned_wavs")
os.makedirs(CLEAN_DIR, exist_ok=True)
print(f"Created clean directory: {CLEAN_DIR}")

manifests = []
# Ensure these columns exist and types are correct before loop
audio_df['Filepath'] = audio_df['Filepath'].apply(lambda p: os.path.normpath(p).replace('\\', '/'))
audio_df['Class'] = audio_df['Class'].astype(int) 
audio_df['Age'] = audio_df['Age'].astype(int)
audio_df['Sex'] = audio_df['Sex'].astype(str)

# Check for missing files before processing (skipped here for brevity, assume previous run handled or fixed)

# Process and re-encode all files
for _, row in tqdm(audio_df.iterrows(), total=len(audio_df), desc="Standardizing Train Audio"):
    src = row['Filepath']
    try:
        # Load and resample to 8kHz mono
        y, sr_ = librosa.load(src, sr=8000, mono=True)
        duration = float(len(y) / 8000.0)
        
        # Define output path and write PCM_16 8kHz mono
        out_name = f"{row['ID']}_{row['Task']}.wav"
        out_path = os.path.join(CLEAN_DIR, out_name)
        sf.write(out_path, y, 8000, subtype='PCM_16')
        
        # Compute SHA256 checksum
        h = hashlib.sha256()
        with open(out_path, 'rb') as fh:
            for chunk in iter(lambda: fh.read(8192), b""):
                h.update(chunk)
        sha256 = h.hexdigest()
        
        manifests.append({
            'ID': row['ID'], 'Task': row['Task'], 'Class': int(row['Class']), 
            'Age': int(row['Age']), 'Sex': row['Sex'], 'OriginalPath': src,
            'RelPath': os.path.relpath(out_path, BASE_DIR).replace('\\', '/'),
            'Duration_s': round(duration, 4),
            'Samplerate': 8000, 'Channels': 1, 'Subtype': 'PCM_16', 
            'SHA256': sha256
        })
    except Exception as e:
        print(f"Failed to process {src}: {e}")

manifest_df = pd.DataFrame(manifests)
manifest_fp = os.path.join(BASE_DIR, "manifest.csv")
clean_meta_fp = os.path.join(BASE_DIR, "cleaned_metadata.csv")

# Save manifests
manifest_df.to_csv(manifest_fp, index=False)
manifest_df[['ID','Age','Sex','Class','Task','RelPath','Duration_s']].to_csv(clean_meta_fp, index=False)

print(f"\n✅ Manifest successfully rebuilt and saved: {manifest_fp}")

In [5]:
# --- 6. Voice Activity Detection (VAD) Helper Function ---

def detect_speech_intervals(y, sr, frame_ms=30, hop_ms=10,
                            min_speech_ms=200, min_silence_ms=150,
                            energy_multiplier=3.0):
    """Simple energy-based VAD with smoothing. Returns list of (start_s, end_s) speech intervals."""
    frame_len = max(2, int(sr * frame_ms / 1000))
    hop_len = max(1, int(sr * hop_ms / 1000))
    
    # RMS energy calculation
    rms = librosa.feature.rms(y=y, frame_length=frame_len, hop_length=hop_len)[0]
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop_len, n_fft=frame_len)
    
    # Robust threshold: median + energy_multiplier * MAD
    med = np.median(rms)
    mad = np.median(np.abs(rms - med)) if len(rms) > 0 else 0.0
    thresh = med + energy_multiplier * mad
    if thresh <= 0:
        thresh = np.percentile(rms, 50) + 1e-9 # Fallback threshold
    mask = rms >= thresh

    # Smoothing (Fill short silences / Remove short speech bursts)
    min_speech_frames = max(1, int(np.ceil(min_speech_ms / hop_ms)))
    min_silence_frames = max(1, int(np.ceil(min_silence_ms / hop_ms)))

    # Fill short silences (bridge)
    i = 0
    while i < len(mask):
        if not mask[i]:
            j = i
            while j < len(mask) and not mask[j]: j += 1
            gap = j - i
            if 0 < gap <= min_silence_frames and i > 0 and j < len(mask):
                mask[i:j] = True
            i = j
        else:
            i += 1

    # Remove short speech segments
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]: j += 1
            length = j - i
            if length < min_speech_frames:
                mask[i:j] = False
            i = j
        else:
            i += 1

    # Convert mask to time intervals (seconds)
    intervals = []
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]: j += 1
            start = times[i]
            # End time approximation
            end = times[min(j - 1, len(times) - 1)] + frame_ms / 1000.0
            intervals.append((max(0.0, float(start)), float(end)))
            i = j
        else:
            i += 1
    return intervals

print("✅ VAD function defined (detect_speech_intervals).")
# Note: You need to re-run the previous cell (Train Audio Cleaning) 
# with `import hashlib` added at the top to correctly generate `manifest_df` 
# before proceeding with the VAD step.

✅ VAD function defined (detect_speech_intervals).


In [8]:
# --- 7. Apply VAD and Extract Timing Features to Manifest ---
SAVED_VAD_CSV = os.path.join(BASE_DIR, "manifest_with_vad.csv")

# Initialize VAD columns if they don't exist
vad_cols = ['speech_intervals', 'speech_duration_s', 'speech_ratio',
            'num_speech_segments', 'mean_speech_segment_s', 'mean_silence_between_s']
for col in vad_cols:
    if col not in manifest_df.columns:
        manifest_df[col] = None

# Process each cleaned file
for idx, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Running VAD on Train Data"):
    rel = row['RelPath']
    full = os.path.normpath(os.path.join(BASE_DIR, rel))
    
    try:
        y, sr = librosa.load(full, sr=8000, mono=True)
        intervals = detect_speech_intervals(y, sr)
        
        speech_durs = [e - s for s, e in intervals]
        total_speech = float(np.sum(speech_durs)) if speech_durs else 0.0
        duration = float(row.get('Duration_s', len(y) / sr if sr > 0 else 0.0))
        speech_ratio = total_speech / duration if duration > 0 else 0.0
        num_segments = len(speech_durs)
        mean_seg = float(np.mean(speech_durs)) if speech_durs else 0.0
        
        mean_sil = 0.0
        if num_segments > 1:
            silences = [intervals[i][0] - intervals[i-1][1] for i in range(1, len(intervals))]
            mean_sil = float(np.mean(silences)) if silences else 0.0

        manifest_df.at[idx, 'speech_intervals'] = intervals
        manifest_df.at[idx, 'speech_duration_s'] = round(total_speech, 4)
        manifest_df.at[idx, 'speech_ratio'] = round(speech_ratio, 4)
        manifest_df.at[idx, 'num_speech_segments'] = int(num_segments)
        manifest_df.at[idx, 'mean_speech_segment_s'] = round(mean_seg, 4)
        manifest_df.at[idx, 'mean_silence_between_s'] = round(mean_sil, 4)
        
    except Exception as e:
        print(f"VAD failed for {full}: {e}")
        # On failure, keep columns as None or 0.0

manifest_df.to_csv(SAVED_VAD_CSV, index=False)
print("\n✅ VAD processing complete.")
print(f"💾 Manifest with VAD saved to: {SAVED_VAD_CSV}")

# Display VAD summary (aggregated diagnostics)
diag = manifest_df.groupby('Class').agg(
    samples=('ID','count'),
    mean_speech_ratio=('speech_ratio','mean'),
    mean_num_segments=('num_speech_segments','mean'),
).round(4)
print("\n--- VAD Summary (Per-Class Averages) ---")
print(diag)

Running VAD on Train Data: 100%|██████████| 2176/2176 [00:04<00:00, 469.21it/s]



✅ VAD processing complete.
💾 Manifest with VAD saved to: Train/task1\manifest_with_vad.csv

--- VAD Summary (Per-Class Averages) ---
       samples  mean_speech_ratio  mean_num_segments
Class                                               
1           48             0.1869             2.1042
2          208             0.1266             1.5337
3          456             0.0906             2.5570
4          608             0.1119             3.3322
5          856             0.1374             2.9416


In [9]:
# --- 8. RMS Volume Normalization (-20 dBFS Target) ---
import hashlib
import shutil # Needed for copying silent files

NORMALIZED_DIR = os.path.join(BASE_DIR, "normalized_wavs")
os.makedirs(NORMALIZED_DIR, exist_ok=True)
print(f"Created normalized directory: {NORMALIZED_DIR}")

TARGET_DBFS = -20.0
TARGET_RMS = 10.0 ** (TARGET_DBFS / 20.0)

# Prepare new columns (re-initialized)
for col in ('NormRelPath', 'NormSHA256', 'NormRMS_db', 'NormPeak', 'NormScale'):
    if col not in manifest_df.columns:
        manifest_df[col] = None

num_scaled_by_peak = 0
orig_rms_db_list = []
norm_rms_db_list = []

for idx, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Normalizing Train Audio"):
    rel = row['RelPath']
    src_fp = os.path.normpath(os.path.join(BASE_DIR, rel))
    
    try:
        # Read as float32 (-1..1 range)
        y, sr = sf.read(src_fp, dtype='float32')
        if y.ndim > 1: y = np.mean(y, axis=1) # Ensure mono
        
        rms = float(np.sqrt(np.mean(np.square(y)))) if y.size > 0 else 0.0
        orig_db = 20.0 * np.log10(rms + 1e-12)
        orig_rms_db_list.append(orig_db)
        
        scale = 1.0 # Default scale
        
        if rms > 1e-6: # Avoid dividing by near-zero RMS
            scale = TARGET_RMS / rms
            peak = float(np.max(np.abs(y)))
            
            # Clip safeguard: re-calculate scale if clipping occurs
            if peak * scale > 0.999:
                scale = min(scale, 0.999 / peak)
                num_scaled_by_peak += 1
            
            y_norm = (y * scale).astype('float32')
            
            norm_rms = float(np.sqrt(np.mean(np.square(y_norm))))
            norm_db = 20.0 * np.log10(norm_rms + 1e-12)
            norm_peak = float(np.max(np.abs(y_norm)))
            norm_rms_db_list.append(norm_db)

            out_name = f"{row['ID']}_{row['Task']}_norm.wav"
            out_path = os.path.join(NORMALIZED_DIR, out_name)
            sf.write(out_path, y_norm, sr, subtype='PCM_16')
            
            sha256 = hashlib.sha256(open(out_path, 'rb').read()).hexdigest()

            manifest_df.at[idx, 'NormRelPath'] = os.path.relpath(out_path, BASE_DIR).replace('\\', '/')
            manifest_df.at[idx, 'NormSHA256'] = sha256
            manifest_df.at[idx, 'NormRMS_db'] = round(norm_db, 4)
            manifest_df.at[idx, 'NormPeak'] = round(norm_peak, 6)
            manifest_df.at[idx, 'NormScale'] = round(float(scale), 6)
            
        else: # Handle silent/near-silent files by copying original (no scaling)
            out_name = f"{row['ID']}_{row['Task']}_norm.wav"
            out_path = os.path.join(NORMALIZED_DIR, out_name)
            shutil.copy2(src_fp, out_path)
            
            manifest_df.at[idx, 'NormRelPath'] = os.path.relpath(out_path, BASE_DIR).replace('\\', '/')
            manifest_df.at[idx, 'NormSHA256'] = hashlib.sha256(open(out_path, 'rb').read()).hexdigest()
            manifest_df.at[idx, 'NormRMS_db'] = round(orig_db, 4)
            manifest_df.at[idx, 'NormPeak'] = float(np.max(np.abs(y)))
            manifest_df.at[idx, 'NormScale'] = 0.0 # No scaling applied

    except Exception as e:
        print(f"Normalization failed for {src_fp}: {e}")

NORMALIZED_MANIFEST = os.path.join(BASE_DIR, "manifest_normalized.csv")
manifest_df.to_csv(NORMALIZED_MANIFEST, index=False)
print(f"\n✅ Files scaled down due to peak limit: {num_scaled_by_peak}")
print(f"💾 Train Manifest (Normalized) saved to: {NORMALIZED_MANIFEST}")

Created normalized directory: Train/task1\normalized_wavs


Normalizing Train Audio: 100%|██████████| 2176/2176 [00:21<00:00, 102.38it/s]


✅ Files scaled down due to peak limit: 565
💾 Train Manifest (Normalized) saved to: Train/task1\manifest_normalized.csv
